### Description:

Calculate the clustering result based on the assignment probabilities obtained from retrieve inference for each training sample (Population Database), and get the statistical info and cases list of each cluster.

>  data structure in `train_assignment_path`
>
>  train_assignment_path
> 
>├── assign
>
>│   ├── SD3434_OS_0.npz
>
>│   ├── SD3434_OS_1.npz
>
>│   ├── SD3434_OS_2.npz
>
>│   ├── SD3463_OS_0.npz
>
>│   ...
 

#### 1. Part1: build up offline bank - get assigment for each training samples

In [1]:
import os
import numpy as np
import pandas as pd

In [3]:
def process_train_assigment(train_assignment_path):
   """
   Process assignment probability files and generate classification results.
   
   Args:
       train_assignment_path (str): Path containing 'assign' folder with .npz files.
                                  Each .npz has 'id' and 'z_1' fields for patient ID
                                  and class probabilities.
   
   Returns:
       tuple: (assigment_result, assignment_prob)
           - assigment_result (pd.DataFrame): Top 1-5 predicted classes and probabilities
           - assignment_prob (pd.DataFrame): Raw probabilities for all classes
   """
    
   # Read all npz files
   assign_dir = os.path.join(train_assignment_path, "assign")
   npz_files = [f for f in os.listdir(assign_dir) if f.endswith('.npz')]
   
   # Get first file to determine number of classes
   first_data = np.load(os.path.join(assign_dir, npz_files[0]))
   n_classes = len(first_data['z_1'])
   
   # Initialize dataframes
   prob_cols = [f'cls_{i}_prob' for i in range(n_classes)]
   assignment_prob = pd.DataFrame(columns=['case_name'] + prob_cols)
   assigment_result = pd.DataFrame(columns=['case_name'] + [f'Top{i}-cls' for i in range(1,6)] + [f'Top{i}-prob' for i in range(1,6)])

   # Process each file
   for npz_file in npz_files:
       data = np.load(os.path.join(assign_dir, npz_file))
       case_name = str(data['id'])
       probs = data['z_1']
       
       # Add to assignment_prob
       row_data = {'case_name': case_name}
       row_data.update({f'cls_{i}_prob': probs[i] for i in range(n_classes)})
       assignment_prob = pd.concat([assignment_prob, pd.DataFrame([row_data])], ignore_index=True)
       
       # Get top 5 classes and probabilities
       top5_idx = np.argsort(probs)[-5:][::-1]
       row_data = {
           'case_name': case_name,
           **{f'Top{i+1}-cls': idx for i, idx in enumerate(top5_idx)},
           **{f'Top{i+1}-prob': probs[idx] for i, idx in enumerate(top5_idx)}
       }
       assigment_result = pd.concat([assigment_result, pd.DataFrame([row_data])], ignore_index=True)

   return assigment_result, assignment_prob


In [ ]:
# get assigment for prototype
train_assignment_path = "result_retrieve/PMQM-retrieve-train"  # path to retrive result on training set
out_train_assigment_path = "retrieve/offlinebank_assigment"
os.makedirs(out_train_assigment_path, exist_ok=True)
result_df, prob_df = process_train_assigment(train_assignment_path)

# Save to Excel
result_df.to_excel(os.path.join(out_train_assigment_path, "train_assignment_result.xlsx"), index=False)
prob_df.to_excel(os.path.join(out_train_assigment_path, "train_assignment_prob.xlsx"), index=False)


/tmp/ipykernel_838187/2749246702.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  assignment_prob = pd.concat([assignment_prob, pd.DataFrame([row_data])], ignore_index=True)
/tmp/ipykernel_838187/2749246702.py:47: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  assigment_result = pd.concat([assigment_result, pd.DataFrame([row_data])], ignore_index=True)


#### 2. Part2: get statistic info of each assigment and get sample list of each cluster

In [5]:
import pandas as pd
import os

def get_cases_of_assignments(result_df, save_dir):
    """
    Groups samples with the same Top1-cls and saves detailed information to different sheets in Excel files.
    Also generates a statistics Excel file.
    
    Parameters:
    -----------
    result_df : pandas.DataFrame
        DataFrame containing case_name, Top1-5 cls and prob information
    save_dir : str
        Directory path to save the output files
        
    Returns:
    --------
    None
    """
    # Ensure save directory exists
    os.makedirs(save_dir, exist_ok=True)
    
    # Define file paths for detailed analysis and statistics
    cases_file = os.path.join(save_dir, 'cases_of_assignment.xlsx')
    stats_file = os.path.join(save_dir, 'stat_each_assignment.xlsx')
    
    # Create list to store statistics data
    stats_data = []
    
    # Create ExcelWriter object for detailed analysis
    with pd.ExcelWriter(cases_file) as writer:
        # Group by Top1-cls
        cls_counts = result_df['Top1-cls'].value_counts().sort_index()
        
        for cls in cls_counts.index:
            # Get all samples for current class
            cls_df = result_df[result_df['Top1-cls'] == cls].copy()
            
            # Collect statistics
            stats_data.append({
                'Class': cls,
                'Count': len(cls_df),
                'Percentage': f"{(len(cls_df) / len(result_df) * 100):.2f}%",
                'Avg_Top1_Prob': f"{cls_df['Top1-prob'].mean():.4f}",
                'Min_Top1_Prob': f"{cls_df['Top1-prob'].min():.4f}",
                'Max_Top1_Prob': f"{cls_df['Top1-prob'].max():.4f}"
            })
            
            # Select required columns
            selected_columns = [
                'case_name',
                'Top1-prob', 'Top2-prob', 'Top3-prob', 'Top4-prob', 'Top5-prob',
                'Top2-cls', 'Top3-cls', 'Top4-cls', 'Top5-cls'
            ]
            
            # Extract needed columns and sort
            output_df = cls_df[selected_columns].sort_values('Top1-prob', ascending=False)
            
            # Save DataFrame to corresponding sheet
            sheet_name = f'Class_{cls}'
            output_df.to_excel(writer, sheet_name=sheet_name, index=False)
            
            # Adjust column widths
            worksheet = writer.sheets[sheet_name]
            for idx, col in enumerate(output_df.columns):
                max_length = max(
                    output_df[col].astype(str).apply(len).max(),
                    len(col)
                ) + 2
                worksheet.column_dimensions[chr(65 + idx)].width = max_length
    
    # Create and save statistics Excel
    stats_df = pd.DataFrame(stats_data)
    
    # Add total row
    total_row = {
        'Class': 'Total',
        'Count': stats_df['Count'].sum(),
        'Percentage': '100.00%',
        'Avg_Top1_Prob': f"{result_df['Top1-prob'].mean():.4f}",
        'Min_Top1_Prob': f"{result_df['Top1-prob'].min():.4f}",
        'Max_Top1_Prob': f"{result_df['Top1-prob'].max():.4f}"
    }
    stats_df = pd.concat([stats_df, pd.DataFrame([total_row])], ignore_index=True)
    
    # Save statistics
    with pd.ExcelWriter(stats_file) as writer:
        stats_df.to_excel(writer, index=False, sheet_name='Statistics')
        worksheet = writer.sheets['Statistics']
        
        # Adjust column widths
        for idx, col in enumerate(stats_df.columns):
            max_length = max(
                stats_df[col].astype(str).apply(len).max(),
                len(col)
            ) + 2
            worksheet.column_dimensions[chr(65 + idx)].width = max_length


In [ ]:
out_train_assigment_path = "retrieve/offlinebank_assigment"
os.makedirs(out_train_assigment_path, exist_ok=True)
get_cases_of_assignments(result_df, save_dir=out_train_assigment_path)